# 🏥 Family Health Guardian: Multi-Disease Machine Learning Pipeline
### Train a 41-Disease, 132-Symptom Diagnostic Model with Kaggle & Colab

This notebook automates downloading the **Disease Symptom Prediction Dataset** from Kaggle using your `kaggle.json`, training an ensemble Random Forest multi-class diagnostic classifier across **41 diseases**, evaluating performance, and downloading the ready-to-deploy model artifacts directly to your computer.

**Pipeline Overview:**
1. 📥 Upload `kaggle.json` & Download Dataset
2. 🧹 Data Cleaning, Standardization & 132-Symptom Binary Encoding
3. 🧠 Train Random Forest Multi-Class Diagnostic Classifier
4. 📊 Comprehensive Evaluation (Accuracy, Precision, Recall, F1-Score)
5. 🧪 Interactive Live Inference Test Function
6. 📦 Export `multi_disease_model.pkl`, `symptoms_list.json`, and `disease_metadata.json`
7. 💾 One-Click Download for Backend Integration


## Step 1: Install Required Packages

In [ ]:
!pip install -q kaggle scikit-learn pandas numpy joblib matplotlib seaborn
print("✅ Packages installed successfully!")

## Step 2: Upload `kaggle.json` & Set Up Kaggle API
Click **Choose Files** below and upload your `kaggle.json` file from your computer.

In [ ]:
import os
from google.colab import files

print("👉 Please select and upload your 'kaggle.json' file:")
uploaded = files.upload()

if 'kaggle.json' in uploaded:
    !mkdir -p ~/.kaggle
    !cp kaggle.json ~/.kaggle/
    !chmod 600 ~/.kaggle/kaggle.json
    print("\n✅ kaggle.json configured successfully!")
else:
    print("\n⚠️ 'kaggle.json' was not uploaded. Please run this cell again and choose kaggle.json.")

## Step 3: Download & Unpack the Disease Symptom Dataset
Downloads `itachi9604/disease-symptom-description-dataset` containing 41 diseases and 132 symptoms.

In [ ]:
!kaggle datasets download -d itachi9604/disease-symptom-description-dataset --force
!unzip -o disease-symptom-description-dataset.zip -d disease_data

print("\n📁 Downloaded files in disease_data/:")
!ls -lh disease_data

## Step 4: Load & Inspect the Datasets

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

data_dir = Path("disease_data")

df_dataset = pd.read_csv(data_dir / "dataset.csv")
df_desc = pd.read_csv(data_dir / "symptom_Description.csv")
df_prec = pd.read_csv(data_dir / "symptom_precaution.csv")
df_sev = pd.read_csv(data_dir / "Symptom-severity.csv")

print(f"Raw dataset shape: {df_dataset.shape}")
print(f"Unique diseases count: {df_dataset['Disease'].nunique()}")
print("\nSample diseases:", list(df_dataset['Disease'].unique()[:10]))
display(df_dataset.head(3))

## Step 5: Data Cleaning & 132-Symptom Binary Encoding
Transform the wide symptom columns (`Symptom_1` to `Symptom_17`) into a standardized binary matrix where:
- Each row = 1 patient record
- Each column = 1 unique symptom (1 if present, 0 if absent)
- Target column = `Disease`

In [ ]:
# Clean strings: strip whitespaces and convert to snake_case
symptom_cols = [c for c in df_dataset.columns if c.startswith('Symptom')]

unique_symptoms = set()
for c in symptom_cols:
    clean_series = df_dataset[c].dropna().astype(str).str.strip().str.replace(' ', '_').str.lower()
    unique_symptoms.update(clean_series.unique())

# Remove empty or invalid tokens
unique_symptoms = sorted([s for s in unique_symptoms if s and s != 'nan'])
print(f"Total unique symptoms identified: {len(unique_symptoms)}")

# Construct binary matrix
binary_records = []
diseases = []

for _, row in df_dataset.iterrows():
    disease = str(row['Disease']).strip()
    row_symptoms = set()
    for c in symptom_cols:
        val = row[c]
        if pd.notna(val):
            cleaned = str(val).strip().replace(' ', '_').lower()
            row_symptoms.add(cleaned)
    
    record = {sym: (1 if sym in row_symptoms else 0) for sym in unique_symptoms}
    binary_records.append(record)
    diseases.append(disease)

X_df = pd.DataFrame(binary_records)
y_series = pd.Series(diseases, name="Disease")

print(f"\nCleaned Feature Matrix Shape: {X_df.shape}")
print(f"Samples per disease distribution (top 5):\n{y_series.value_counts().head(5)}")

## Step 6: Stratified Train / Test Split (80% Train, 20% Test)
We use Stratified Splitting to ensure all 41 diseases are equally represented in both partitions without leakage.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_df, y_series,
    test_size=0.20,
    random_state=42,
    stratify=y_series
)

print(f"Training set shape: {X_train.shape} ({len(y_train)} samples)")
print(f"Test set shape:     {X_test.shape} ({len(y_test)} samples)")

## Step 7: Train the Random Forest Multi-Class Classifier

In [ ]:
from sklearn.ensemble import RandomForestClassifier
import time

print("Training Random Forest (100 estimators) across 41 disease classes...")
t0 = time.time()

model = RandomForestClassifier(
    n_estimators=100,
    criterion='gini',
    max_depth=None,
    random_state=42,
    n_jobs=-1
)
model.fit(X_train, y_train)
elapsed = time.time() - t0

print(f"✅ Model trained in {elapsed:.2f} seconds!")

## Step 8: Comprehensive Model Evaluation

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

y_pred = model.predict(X_test)

acc = accuracy_score(y_test, y_pred)
prec_macro = precision_score(y_test, y_pred, average='macro', zero_division=0)
rec_macro = recall_score(y_test, y_pred, average='macro', zero_division=0)
f1_macro = f1_score(y_test, y_pred, average='macro', zero_division=0)

print("=" * 60)
print("           MULTI-DISEASE MODEL EVALUATION REPORT           ")
print("=" * 60)
print(f"Test Accuracy:       {acc * 100:.2f}%")
print(f"Macro Precision:     {prec_macro * 100:.2f}%")
print(f"Macro Recall:        {rec_macro * 100:.2f}%")
print(f"Macro F1-Score:      {f1_macro * 100:.2f}%")
print("=" * 60)

print("\nDetailed Classification Report (Sample classes):")
print(classification_report(y_test, y_pred, digits=4, zero_division=0))

## Step 9: Package Clinical Descriptions & Precautions Encyclopedia

In [ ]:
import json

# Clean description lookup
desc_dict = {}
for _, row in df_desc.iterrows():
    d_name = str(row['Disease']).strip()
    desc_dict[d_name] = str(row['Description']).strip()

# Clean precautions lookup
prec_dict = {}
for _, row in df_prec.iterrows():
    d_name = str(row['Disease']).strip()
    p_list = []
    for p_col in ['Precaution_1', 'Precaution_2', 'Precaution_3', 'Precaution_4']:
        val = row.get(p_col)
        if pd.notna(val) and str(val).strip():
            p_list.append(str(val).strip().capitalize())
    prec_dict[d_name] = p_list

# Aggregate disease metadata
disease_metadata = {}
all_classes = sorted(list(model.classes_))

for d in all_classes:
    disease_metadata[d] = {
        "disease_name": d,
        "description": desc_dict.get(d, "Clinical condition characterized by specified symptom complex."),
        "precautions": prec_dict.get(d, ["Consult a qualified healthcare physician", "Follow prescribed care protocol", "Monitor symptoms"])
    }

print(f"✅ Built clinical metadata encyclopedia for {len(disease_metadata)} diseases.")
print("\nExample (Malaria):")
print(json.dumps(disease_metadata.get('Malaria', {}), indent=2))

## Step 10: Live Interactive Symptom Prediction Function
Test the model with arbitrary lists of symptoms!

In [ ]:
def predict_disease(input_symptoms, top_k=3):
    # Clean and validate input symptoms
    cleaned_input = [s.strip().replace(' ', '_').lower() for s in input_symptoms]
    
    # Vectorize
    vec = np.zeros(len(unique_symptoms), dtype=int)
    matched_symptoms = []
    for idx, sym in enumerate(unique_symptoms):
        if sym in cleaned_input:
            vec[idx] = 1
            matched_symptoms.append(sym)
            
    if not matched_symptoms:
        return {"error": "No recognized symptoms provided in input list."}
        
    input_df = pd.DataFrame([vec], columns=unique_symptoms)
    probabilities = model.predict_proba(input_df)[0]
    
    # Top-k ranking
    top_indices = np.argsort(probabilities)[::-1][:top_k]
    
    results = []
    for i in top_indices:
        d_name = model.classes_[i]
        prob = float(probabilities[i])
        meta = disease_metadata.get(d_name, {})
        results.append({
            "disease": d_name,
            "confidence_percentage": round(prob * 100.0, 1),
            "description": meta.get("description", ""),
            "precautions": meta.get("precautions", [])
        })
        
    return {
        "matched_symptoms": matched_symptoms,
        "predictions": results
    }

# --- Test Case 1: Malaria Symptoms ---
print("\n🧪 TEST CASE 1: Malaria")
res1 = predict_disease(['chills', 'vomiting', 'high_fever', 'sweating', 'headache', 'nausea'])
print(json.dumps(res1, indent=2))

# --- Test Case 2: Pneumonia Symptoms ---
print("\n🧪 TEST CASE 2: Pneumonia")
res2 = predict_disease(['cough', 'high_fever', 'breathlessness', 'chest_pain'])
print(json.dumps(res2, indent=2))

# --- Test Case 3: Dermatology / Fungal Infection ---
print("\n🧪 TEST CASE 3: Fungal Infection")
res3 = predict_disease(['itching', 'skin_rash', 'nodal_skin_eruptions'])
print(json.dumps(res3, indent=2))

## Step 11: Save Artifacts & Download for Backend
This step saves all 3 required files and zips them into `multi_disease_artifacts.zip`, then automatically downloads it to your computer!

In [ ]:
import joblib
import zipfile
from google.colab import files

output_dir = Path("exported_artifacts")
output_dir.mkdir(exist_ok=True)

# 1. Save Scikit-Learn Model
model_path = output_dir / "multi_disease_model.pkl"
joblib.dump(model, model_path, compress=3)
print(f"Saved model to: {model_path} ({model_path.stat().st_size / 1024:.1f} KB)")

# 2. Save Symptoms List (exact feature ordering for API)
symptoms_path = output_dir / "symptoms_list.json"
with open(symptoms_path, "w") as f:
    json.dump({
        "total_symptoms": len(unique_symptoms),
        "symptoms": unique_symptoms
    }, f, indent=2)
print(f"Saved symptoms list to: {symptoms_path}")

# 3. Save Disease Metadata & Precautions
meta_path = output_dir / "disease_metadata.json"
with open(meta_path, "w") as f:
    json.dump({
        "model_name": "RandomForestClassifier",
        "total_diseases": len(disease_metadata),
        "evaluation_accuracy": round(acc, 4),
        "evaluation_f1_macro": round(f1_macro, 4),
        "diseases": disease_metadata
    }, f, indent=2)
print(f"Saved disease metadata to: {meta_path}")

# 4. Create ZIP bundle
zip_path = "multi_disease_artifacts.zip"
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
    zipf.write(model_path, arcname="multi_disease_model.pkl")
    zipf.write(symptoms_path, arcname="symptoms_list.json")
    zipf.write(meta_path, arcname="disease_metadata.json")

print(f"\n🎁 Created bundle: {zip_path} ({os.path.getsize(zip_path) / 1024:.1f} KB)")
print("\n⬇️ Downloading multi_disease_artifacts.zip to your computer now...")
files.download(zip_path)